# 06 — Forecasting
Revenue forecasting with multiple models and backtesting.

In [ ]:
import json
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

PROCESSED = Path('..') / 'data' / 'processed'
forecast = pd.read_csv(PROCESSED / 'forecast_30d.csv', parse_dates=['date'])
report = json.loads((PROCESSED / 'forecast_report.json').read_text())
print(f'Forecast days: {len(forecast)}')
report

## 30-Day Forecast Comparison

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=forecast['date'], y=forecast['xgboost'], name='XGBoost', mode='lines'))
fig.add_trace(go.Scatter(x=forecast['date'], y=forecast['naive'], name='Naive', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=forecast['date'], y=forecast['moving_avg_7d'], name='Moving Avg', line=dict(dash='dot')))
fig.update_layout(title='30-Day Revenue Forecast', yaxis_title='Revenue (£)')
fig.show()

## Model Performance

In [ ]:
xgb = report.get('xgboost', {})
print('XGBoost Performance:')
print(f'  MAE:  £{xgb.get("mae", 0):,.0f}')
print(f'  RMSE: £{xgb.get("rmse", 0):,.0f}')
print(f'  MAPE: {xgb.get("mape", 0):.1f}%')

summary = report.get('forecast_summary', {})
print(f'\n30-Day Forecast Total: £{summary.get("xgboost_total_forecast", 0):,.0f}')
print(f'Avg Daily Forecast:   £{summary.get("xgboost_avg_forecast", 0):,.0f}')

## Forecast Summary Statistics

In [ ]:
for col in ['xgboost', 'naive', 'moving_avg_7d']:
    print(f'{col}: mean=£{forecast[col].mean():,.0f}, total=£{forecast[col].sum():,.0f}')

## Methodology

- **Naive**: Last value carried forward
- **Moving Average**: 7-day rolling mean
- **XGBoost**: Lag features (1,2,3,7,14,28d) + rolling stats (7,14,28d)
- Train/test split: 80/20 temporal (no shuffling)
- Forecast: recursive 30-day ahead